# VGG16 4-bit with a squeezed 8x8 layer

`features[27]` is an 8-in / 8-out `QuantConv2d` with no BatchNorm (conv -> ReLU), so it maps onto the 8x8 systolic array without tiling. `features[24]` is 256 -> 8 and `features[29]` is 8 -> 512.

**Order:** run setup (1), load pretrained weights (2), train (3), evaluate (4), then verification (5-9). To skip training, run 1, then 4 onward.

## 1. Setup

In [1]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

import torchvision
import torchvision.transforms as transforms

from models import *

global best_prec
use_gpu = torch.cuda.is_available()
print('=> Building model...')
    
    
batch_size = 128
model_name = "VGG16_quant_squeeze"
model = VGG16_quant_squeeze()
model.features[0].act_alpha.data.fill_(1.0)
print(model)

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)

        input, target = input.cuda(), target.cuda()

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.cuda(), target.cuda()

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [150, 225]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1        

#model = nn.DataParallel(model).cuda()
#all_params = checkpoint['state_dict']
#model.load_state_dict(all_params, strict=False)
#criterion = nn.CrossEntropyLoss().cuda()
#validate(testloader, model, criterion)

=> Building model...
VGG_quant(
  (features): Sequential(
    (0): QuantConv2d(
      3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): QuantConv2d(
      64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): QuantConv2d(
      64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): QuantConv2d(
      128, 128, kernel_size=(3, 3), stride

## 2. Initialize from the 92% 4-bit checkpoint

Layer indices shift because one BatchNorm was removed, so layers are matched by order (the 13 conv/BN pairs) and copied wherever the shapes match. Only the three resized layers start from random weights.

In [2]:
# Import explicitly: after `from models import *`, the name QuantConv2d refers to the 2-bit class
# from quant_layer_2bit (imported by resnet20_quant_2bit), not the one VGG16_quant uses.
from models.quant_layer import QuantConv2d

PRETRAINED = "result/VGG16_quant_4bit/model_best.pth.tar"

def conv_bn_pairs(m):
    mods = list(m.features)
    pairs = []
    for i, mod in enumerate(mods):
        if isinstance(mod, QuantConv2d):
            nxt = mods[i + 1]
            pairs.append((i, mod, nxt if isinstance(nxt, nn.BatchNorm2d) else None))
    return pairs

src_model = VGG16_quant()
src_model.load_state_dict(torch.load(PRETRAINED, map_location="cpu")["state_dict"])

src_pairs, dst_pairs = conv_bn_pairs(src_model), conv_bn_pairs(model)
assert len(src_pairs) == len(dst_pairs) == 13
for (_, s_conv, s_bn), (i, d_conv, d_bn) in zip(src_pairs, dst_pairs):
    if s_conv.weight.shape == d_conv.weight.shape:
        d_conv.load_state_dict(s_conv.state_dict())
        if s_bn is not None and d_bn is not None:
            d_bn.load_state_dict(s_bn.state_dict())
        print(f"features[{i}]: copied   {tuple(d_conv.weight.shape)}")
    else:
        print(f"features[{i}]: new init {tuple(d_conv.weight.shape)}")
model.classifier.load_state_dict(src_model.classifier.state_dict())
del src_model


features[0]: copied   (64, 3, 3, 3)
features[3]: copied   (64, 64, 3, 3)
features[7]: copied   (128, 64, 3, 3)
features[10]: copied   (128, 128, 3, 3)
features[14]: copied   (256, 128, 3, 3)
features[17]: copied   (256, 256, 3, 3)
features[20]: copied   (256, 256, 3, 3)
features[24]: new init (8, 256, 3, 3)
features[27]: new init (8, 8, 3, 3)
features[29]: new init (512, 8, 3, 3)
features[33]: copied   (512, 512, 3, 3)
features[36]: copied   (512, 512, 3, 3)
features[39]: copied   (512, 512, 3, 3)


## 3. Train (fine-tune)

Watch `best acc` (test accuracy), not the training `Prec`. If it plateaus below 90%, try `lr = 4e-2` and `epochs = 200` (the original recipe). Don't register any hooks before or during training.

In [ ]:
lr = 2e-2
weight_decay = 1e-4
epochs = 100
best_prec = 0

model.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

fdir = 'result/' + str(model_name)
os.makedirs(fdir, exist_ok=True)

for epoch in range(0, epochs):
    train(trainloader, model, criterion, optimizer, epoch)

    print("Validation starts")
    prec = validate(testloader, model, criterion)

    scheduler.step()

    is_best = prec > best_prec
    best_prec = max(prec, best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)


Epoch: [0][0/391]	Time 1.140 (1.140)	Data 0.214 (0.214)	Loss 5.3826 (5.3826)	Prec 5.469% (5.469%)
Epoch: [0][100/391]	Time 0.164 (0.166)	Data 0.004 (0.006)	Loss 0.3585 (0.7332)	Prec 88.281% (75.209%)
Epoch: [0][200/391]	Time 0.154 (0.161)	Data 0.004 (0.005)	Loss 0.2699 (0.5248)	Prec 89.062% (82.521%)
Epoch: [0][300/391]	Time 0.154 (0.159)	Data 0.005 (0.005)	Loss 0.2739 (0.4370)	Prec 90.625% (85.543%)
Validation starts
Test: [0/79]	Time 0.204 (0.204)	Loss 0.3921 (0.3921)	Prec 89.844% (89.844%)
 * Prec 85.660% 
best acc: 85.660000
Epoch: [1][0/391]	Time 0.394 (0.394)	Data 0.251 (0.251)	Loss 0.1307 (0.1307)	Prec 96.875% (96.875%)
Epoch: [1][100/391]	Time 0.147 (0.159)	Data 0.005 (0.006)	Loss 0.2954 (0.1967)	Prec 89.844% (93.619%)
Epoch: [1][200/391]	Time 0.155 (0.157)	Data 0.004 (0.005)	Loss 0.2443 (0.2017)	Prec 92.969% (93.497%)
Epoch: [1][300/391]	Time 0.164 (0.157)	Data 0.005 (0.005)	Loss 0.1671 (0.1997)	Prec 96.094% (93.467%)
Validation starts
Test: [0/79]	Time 0.186 (0.186)	Loss 0.45

## 4. Evaluate the best checkpoint

Success criterion: accuracy > 90%.

In [ ]:
PATH = "result/VGG16_quant_squeeze/model_best.pth.tar"
device = torch.device("cuda")
checkpoint = torch.load(PATH, map_location=device)
model.load_state_dict(checkpoint['state_dict'])
model.to(device)
model.eval()

correct = 0
with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device)
        pred = model(data).argmax(dim=1)
        correct += (pred == target).sum().item()

acc = 100. * correct / len(testloader.dataset)
print(f"Test accuracy: {correct}/{len(testloader.dataset)} ({acc:.2f}%)  ->", "PASS" if acc > 90 else "FAIL")


## 5. Capture inputs of the squeezed layer and the next layer

TF32 is turned off: on Ampere GPUs (including Orin) PyTorch runs convolutions in TF32 by default, and its 10-bit mantissa alone would put the reference off by about 1e-3.

In [ ]:
from models.quant_layer import QuantConv2d, act_quantization

torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False

SQ, NEXT = 27, 29
sq_layer, next_layer = model.features[SQ], model.features[NEXT]
assert isinstance(sq_layer, QuantConv2d) and sq_layer.in_channels == 8 and sq_layer.out_channels == 8
assert isinstance(model.features[SQ + 1], nn.ReLU) and isinstance(next_layer, QuantConv2d)  # conv -> relu, no BN

class SaveOutput:
    def __init__(self):
        self.outputs = []
    def __call__(self, module, module_in):
        self.outputs.append(module_in)
    def clear(self):
        self.outputs = []

save_sq, save_next = SaveOutput(), SaveOutput()
h_sq = sq_layer.register_forward_pre_hook(save_sq)
h_next = next_layer.register_forward_pre_hook(save_next)

model.eval()
images, labels = next(iter(testloader))
with torch.no_grad():
    model(images.to(device))
h_sq.remove(); h_next.remove()   # remove hooks so later forwards don't keep accumulating

x_sq = save_sq.outputs[0][0]       # input to features[27]   [B, 8, 4, 4]
x_next = save_next.outputs[0][0]   # input to features[29] = ReLU(features[27] output)   [B, 8, 4, 4]
print("squeezed layer input:", tuple(x_sq.shape), " next layer input:", tuple(x_next.shape))


## 6. Integer weights and activations of the squeezed layer

In [ ]:
w_bit, act_bit = 4, 4
w_alpha = sq_layer.weight_quant.wgt_alpha
a_alpha = sq_layer.act_alpha
w_scale = (w_alpha / (2 ** (w_bit - 1) - 1)).detach()   # alpha / 7
a_scale = (a_alpha / (2 ** act_bit - 1)).detach()       # alpha / 15

with torch.no_grad():
    weight_int = sq_layer.weight_q / w_scale
    act_int = act_quantization(act_bit)(x_sq, a_alpha) / a_scale

assert torch.allclose(weight_int, weight_int.round(), atol=1e-3), "weights are not integers"
assert torch.allclose(act_int, act_int.round(), atol=1e-3), "activations are not integers"
weight_int, act_int = weight_int.round(), act_int.round()
assert weight_int.min() >= -7 and weight_int.max() <= 7
assert act_int.min() >= 0 and act_int.max() <= 15

print("weight_int:", tuple(weight_int.shape), "range", weight_int.min().item(), "to", weight_int.max().item())
print("act_int:   ", tuple(act_int.shape), "range", act_int.min().item(), "to", act_int.max().item())


## 7. Map onto the 8x8 systolic array (no tiling)

For each kernel position `kij`, the array holds one 8x8 weight slice `w_int[:, :, kij]` (PE at row r, column c holds `w_int[c, r, kij]`) and the 36 padded input positions `nij` are streamed through it.

In [ ]:
import math

array_size = 8
padding, stride = 1, 1

a_int = act_int[0]                                                    # one image: [8, 4, 4]
w_int = weight_int.reshape(weight_int.size(0), weight_int.size(1), -1)  # [oc, ic, kij] = [8, 8, 9]
assert w_int.size(0) == array_size and w_int.size(1) == array_size    # fits the array, no tiling

ic, ni, nj = a_int.shape
a_pad = torch.zeros(ic, ni + 2 * padding, nj + 2 * padding, device=a_int.device)
a_pad[:, padding:padding + ni, padding:padding + nj] = a_int
a_pad_ni_dim = a_pad.size(1)                                          # 6
a_pad = a_pad.reshape(ic, -1)                                         # [8, 36]

kijg = range(w_int.size(2))
ki_dim = int(math.sqrt(w_int.size(2)))                                # 3

psum = torch.zeros(array_size, a_pad.size(1), len(kijg), device=a_int.device)   # [8, 36, 9]
for kij in kijg:
    psum[:, :, kij] = w_int[:, :, kij] @ a_pad                        # one weight load, stream all nij

print("a_pad:", tuple(a_pad.shape), " psum:", tuple(psum.shape))


## 8. Accumulate (SFP) and recover `psum_recovered` with ReLU

In [ ]:
o_ni_dim = (a_pad_ni_dim - ki_dim) // stride + 1                    # 4
o_nijg = range(o_ni_dim ** 2)                                         # 16 output positions

out = torch.zeros(array_size, len(o_nijg), device=a_int.device)       # [8, 16]
for o_nij in o_nijg:
    for kij in kijg:
        out[:, o_nij] += psum[:, (o_nij // o_ni_dim) * a_pad_ni_dim + o_nij % o_ni_dim
                                 + (kij // ki_dim) * a_pad_ni_dim + kij % ki_dim, kij]

psum_recovered = torch.relu(out * a_scale * w_scale)
print("psum_recovered:", tuple(psum_recovered.shape))


## 9. Compare with the next layer's prehooked input

Success criterion: max error < 1e-3 (expected about 1e-5 or less).

In [ ]:
ref = x_next[0].reshape(array_size, -1)                               # [8, 16]
err = (psum_recovered - ref).abs()
print(f"image 0:   max abs error {err.max().item():.3e}, mean {err.mean().item():.3e}")
# float32 has ~7 significant digits, so the smallest possible error scales with the output size
print(f"           largest |output| {ref.abs().max().item():.3f} -> float32 spacing there "
      f"{torch.finfo(torch.float32).eps * ref.abs().max().item():.1e}")

# same check over the whole batch, using conv2d on the integer tensors
with torch.no_grad():
    batch_rec = torch.relu(F.conv2d(act_int, weight_int, padding=padding) * a_scale * w_scale)
batch_err = (batch_rec - x_next).abs().max().item()
print(f"full batch: max abs error {batch_err:.3e}")

print("PASS" if err.max().item() < 1e-3 and batch_err < 1e-3 else "FAIL")
